# Vision-Based Vectorless RAG
**Answer questions from PDF page images — No OCR, No Vector DB**

How it works:
```
1. Build tree from PDF text  →  LLM finds relevant sections
2. Convert those pages to images
3. Vision LLM reads the images and answers directly
```
Uses only: `groq` (free) + `PyPDF2` + `PyMuPDF` + `requests` — no PageIndex API needed.

## Step 0 — Install

In [1]:
!pip install -q groq PyPDF2 PyMuPDF python-dotenv requests


[notice] A new release of pip is available: 25.0.1 -> 26.0.1
[notice] To update, run: C:\Users\Nidhi\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.12_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


## Step 0.1 — Setup

In [2]:
import os, json, base64, requests
import PyPDF2
import fitz   # PyMuPDF — converts PDF pages to images
from groq import Groq
from dotenv import load_dotenv

load_dotenv()
GROQ_API_KEY   = os.getenv("GROQ_API_KEY", "").strip()
TEXT_MODEL     = "llama-3.3-70b-versatile"      # for tree building & search
VISION_MODEL   = "meta-llama/llama-4-scout-17b-16e-instruct"  # for image QA
client         = Groq(api_key=GROQ_API_KEY)

print("Groq:", "OK" if GROQ_API_KEY else "MISSING — add GROQ_API_KEY to .env")
print("Text model: ", TEXT_MODEL)
print("Vision model:", VISION_MODEL)

def call_llm(prompt, response_json=False):
    kwargs = {"response_format": {"type": "json_object"}} if response_json else {}
    res = client.chat.completions.create(
        model=TEXT_MODEL,
        messages=[{"role": "user", "content": prompt}],
        temperature=0, **kwargs
    )
    return res.choices[0].message.content.strip()

Groq: OK
Text model:  llama-3.3-70b-versatile
Vision model: meta-llama/llama-4-scout-17b-16e-instruct


## Step 1 — Download & Load PDF
Using the **"Attention Is All You Need"** paper from arxiv (public, free).

In [3]:
PDF_URL  = "https://arxiv.org/pdf/1706.03762.pdf"
PDF_PATH = "attention_is_all_you_need.pdf"

if not os.path.exists(PDF_PATH):
    print("Downloading paper...")
    r = requests.get(PDF_URL, timeout=60)
    with open(PDF_PATH, "wb") as f:
        f.write(r.content)
    print("Downloaded.")
else:
    print("PDF already exists, skipping download.")

reader = PyPDF2.PdfReader(PDF_PATH)
pages  = [reader.pages[i].extract_text() or "" for i in range(len(reader.pages))]
print(f"PDF loaded: {len(pages)} pages")

Downloaded.


PDF loaded: 15 pages


## Step 1.1 — Build Tree Structure
LLM analyzes the text and builds a hierarchical section tree.

In [4]:
def build_tree(pages):
    def clean(text):
        return text.encode("ascii", "ignore").decode("ascii")

    sample = ""
    for i, text in enumerate(pages[:12], 1):
        sample += f"\n--- Page {i} ---\n{clean(text)[:500]}\n"

    prompt = f"""Analyze this document and extract its section structure as a tree.

Document sample:
{sample}

Return a JSON object with key "tree" containing an array of sections.
Each section must have:
- node_id: string like "0001"
- title: section name string
- page_index: integer page number
- summary: one sentence string
- nodes: array of child sections (empty array if none)

Example: {{"tree": [{{"node_id": "0001", "title": "Abstract", "page_index": 1, "summary": "Overview.", "nodes": []}}]}}"""

    raw    = call_llm(prompt, response_json=True)
    result = json.loads(raw)
    if "tree" in result:
        return result["tree"]
    for v in result.values():
        if isinstance(v, list):
            return v
    return []

print("Building tree... (~10 seconds)")
tree = build_tree(pages)

def print_tree(nodes, indent=0):
    for n in nodes:
        print("  " * indent + f"[{n['node_id']}] {n['title']}  (p.{n.get('page_index','?')})")
        if n.get("nodes"):
            print_tree(n["nodes"], indent + 1)

print(f"Tree built: {len(tree)} top-level sections\n")
print("Document Structure:")
print_tree(tree)

Building tree... (~10 seconds)


Tree built: 4 top-level sections

Document Structure:
[0001] Introduction  (p.2)
  [0002] Encoder and Decoder Stacks  (p.3)
    [0003] Scaled Dot-Product Attention  (p.4)
    [0004] Multi-Head Attention  (p.5)
[0005] Complexity Analysis  (p.6)
[0006] Experimental Results  (p.8)
  [0007] Variations on the Transformer Architecture  (p.9)
  [0008] Generalization to English Constituency Parsing  (p.10)
[0009] References  (p.11)


## Step 2 — Reasoning-Based Retrieval (Tree Search)
LLM picks the relevant nodes from the tree.

In [5]:
query = "What is the last operation in the Scaled Dot-Product Attention figure?"

def compress_tree(nodes):
    return [{"node_id": n["node_id"], "title": n["title"],
             "page": n.get("page_index", "?"),
             "summary": n.get("summary", ""),
             **(({"nodes": compress_tree(n["nodes"])} if n.get("nodes") else {}))}
            for n in nodes]

search_prompt = f"""You are given a question and a document tree (Table of Contents with summaries).
Find all node_ids likely to contain the answer.

Question: {query}

Document tree:
{json.dumps(compress_tree(tree), indent=2)}

Reply ONLY in JSON:
{{"thinking": "<your reasoning>", "node_list": ["0001", "0002"]}}"""

result   = json.loads(call_llm(search_prompt, response_json=True))
node_ids = result["node_list"]

print("Reasoning Process:")
print(result["thinking"])
print("\nRetrieved Node IDs:", node_ids)

Reasoning Process:
To find the last operation in the Scaled Dot-Product Attention figure, we need to look for the node that directly discusses Scaled Dot-Product Attention. The document tree is traversed to find the node with the title 'Scaled Dot-Product Attention'. This node is likely to contain the answer because it directly addresses the topic of interest. Since the question is about the 'last operation' in this context, we also consider the parent node of 'Scaled Dot-Product Attention' as it may provide context or introductory information that could be relevant to understanding the operations involved.

Retrieved Node IDs: ['0003', '0002']


## Step 2.1 — Show Retrieved Nodes

In [6]:
def find_nodes(tree, target_ids):
    found = []
    for n in tree:
        if n["node_id"] in target_ids:
            found.append(n)
        if n.get("nodes"):
            found.extend(find_nodes(n["nodes"], target_ids))
    return found

retrieved = find_nodes(tree, node_ids)
print("Retrieved Nodes:\n")
for n in retrieved:
    print(f"  Node ID: {n['node_id']}   Page: {n.get('page_index','?')}   Title: {n['title']}")

Retrieved Nodes:

  Node ID: 0002   Page: 3   Title: Encoder and Decoder Stacks
  Node ID: 0003   Page: 4   Title: Scaled Dot-Product Attention


## Step 2.2 — Convert Retrieved Pages to Images
PyMuPDF renders the relevant PDF pages into JPEG images (stored in memory — no files saved).

In [7]:
def pages_to_base64_images(pdf_path, page_numbers):
    """Convert specific PDF pages to base64 JPEG images (in memory)."""
    doc    = fitz.open(pdf_path)
    images = []
    seen   = set()
    for pg in sorted(set(page_numbers)):
        if pg in seen or pg < 1 or pg > len(doc):
            continue
        seen.add(pg)
        page = doc.load_page(pg - 1)   # 0-indexed
        pix  = page.get_pixmap(matrix=fitz.Matrix(2.0, 2.0))  # 2x zoom = better quality
        b64  = base64.b64encode(pix.tobytes("jpeg")).decode("utf-8")
        images.append({"page": pg, "b64": b64})
        print(f"  Page {pg} converted to image")
    doc.close()
    return images

# Collect page numbers from retrieved nodes
page_numbers = [n.get("page_index", 1) for n in retrieved]
# Also include the next page for each node (in case content spans pages)
page_numbers += [p + 1 for p in page_numbers]

print("Converting PDF pages to images...")
images = pages_to_base64_images(PDF_PATH, page_numbers)
print(f"\nTotal images ready: {len(images)}")

Converting PDF pages to images...


  Page 3 converted to image

  Page 4 converted to image
  Page 5 converted to image

Total images ready: 3


## Step 3 — Vision LLM Answers from Page Images
The vision model looks at the actual PDF page images and answers — **no OCR needed**.

In [8]:
def call_vision_llm(prompt, images):
    """Send prompt + PDF page images to Groq vision model."""
    content = [{"type": "text", "text": prompt}]
    for img in images:
        content.append({
            "type": "image_url",
            "image_url": {"url": f"data:image/jpeg;base64,{img['b64']}"}
        })
    res = client.chat.completions.create(
        model=VISION_MODEL,
        messages=[{"role": "user", "content": content}],
        temperature=0
    )
    return res.choices[0].message.content.strip()


answer_prompt = f"""Answer the question by looking at the document page images provided.
Give a clear, concise answer based only on what you see in the images.

Question: {query}"""

print(f"Sending {len(images)} page image(s) to vision model...\n")
answer = call_vision_llm(answer_prompt, images)

print("Generated Answer:\n")
print(answer)

Sending 3 page image(s) to vision model...



Generated Answer:

The last operation in the Scaled Dot-Product Attention figure is **Softmax**.
